# Events and Model State

In this lesson, you will learn to connect user actions to protected model state and refresh the visible result.

CSC-239 · Module 12 · Lesson 3 of 4

A useful interface responds after its window appears. We will connect a reservation button to a small Java model, keep the displayed count current, and test the boundary with pointer and keyboard input. Use the [module glossary](terms.md) to revisit the terms after their explanations.


## Learning Goals

- Register a handler that changes an ordinary Java model and updates a visible Label.
- Use pointer and keyboard input to reach a boundary state and verify the button’s disabled behavior.


## Why This Matters

A reservation desk needs a count that remains trustworthy after people use the interface. A visible number can look correct at startup and become wrong after the next action. Keeping the rule in an ordinary Java object lets the program protect its state, while the interface communicates the current result.

The preceding lessons built and arranged visible content. Here, users decide when that content changes. Registering a short action handler lets a button serve pointer and keyboard users through the same model rule. Updating the model, its displayed text, and the button's availability together helps prevent a stale message or an action that appears available after the resource is exhausted.


## Check Your Starting Point

Connect the earlier ideas about encapsulation, lambdas, mutable objects, and JavaFX layout. Explain the relationships before reading the answer.


Explain how a private field and guarded public method protect a count. Explain when a supplied lambda body executes and how it can change an object while keeping its local reference unchanged. Distinguish a VBox child from an ordinary Java model, and state the job of the supplied `Fx.run` operation.


In [ ]:
Private state and guard:
Your response

Delayed lambda and stable reference:
Your response

VBox/model roles and supplied support:
Your response


<details>
<summary>Show answer</summary>

A private field keeps callers from assigning the stored count directly. A public method can check a condition before changing that field. The condition belongs to the model rule, even when an interface also discourages an unavailable action.

Creating or passing a lambda supplies behavior. Its body runs when the operation that receives it invokes it. A stable local reference can still refer to an object whose state changes through methods; changing that object's state does not reassign the local reference.

The VBox contains the visible child nodes. An ordinary Java model stores the application state and does not become a VBox child. The supplied `Fx.run` operation performs short live-interface work on the JavaFX Application Thread.

</details>


## Video Demonstration

Predict the label after each reservation. Watch the native action change the model, update the label, and disable the button at zero. Compare the initial console report with later visible UI state.

<video controls preload="metadata" width="960">
  <source src="media/03_events_and_model_state/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/03_events_and_model_state/captions.vtt" srclang="en" label="English">
  Your browser does not support embedded video.
</video>

[Read the events and model state demonstration transcript](media/03_events_and_model_state/transcript.md).


## Concept

### Give the user a meaningful action

A campus welcome session has three seats left. A staff member needs a window that shows the remaining count and lets them reserve one seat at a time. In this small model, one accepted activation uses one seat. The count must stop at zero. We are modeling one local window, not a shared booking service for multiple users.

A **Button** is a visible control that lets a user request an action. The complete program creates its button with:

```java
    Button reserve = new Button("Reserve one");
```

The constructor creates a button object. Its string argument supplies the text the user sees. The variable `reserve` refers to that button so the program can later connect behavior and change its availability. The words Reserve one describe the action; they are not Java instructions that perform a reservation.

Construction alone does not change the seat count. We must connect the control to behavior that JavaFX can invoke after the window appears. That separation matters because users decide when to act, often long after the initial interface-building code has finished.


### Describe activation independently of the input device

An **action event** is a notification that a control's action has been activated. A pointer click can activate a button. A keyboard user can also activate an enabled button that has keyboard focus. Both routes should request the same reservation rule.

The program responds to the button's action rather than writing one reservation rule for mouse input and another for keyboard input. This gives the control a consistent meaning: Reserve one asks for one reservation, whichever supported input method produced the action.

JavaFX supplies an event object when it delivers that notification. The handler in our example names its parameter `event`. That name represents the notification; it is not the seat count or the button's visible text. This example does not need to inspect information inside the event object. It needs the notification as the reason to perform the registered operation.

An event and a model change are still separate things. The event asks the program to act. The program's model decides what that action may change.


### Register behavior now for JavaFX to call later

**Event-handler registration** connects behavior to a future event. A **callback** is behavior another part of the system calls when the appropriate situation occurs. JavaFX uses the callback registered through `setOnAction` when the button's action occurs.

The complete example constructs a seat model named `model`, a count label named `status`, and the button named `reserve`. The following fragment assumes those objects already exist:

```java
    reserve.setOnAction(event -> {
        model.reserve();
        status.setText("Remaining: " + model.getRemaining());
        reserve.setDisable(model.getRemaining() == 0);
    });
```

The call to `setOnAction` registers the lambda expression as the handler. The parameter `event` is before `->`; the statements between the braces are the work to perform when JavaFX calls that handler. This reuses the lambda syntax taught earlier in the course. We will examine the model and the three statements in the next sections.

Registration does not execute those three statements immediately. In particular, `model.reserve()` is inside the lambda body. Moving that call into the construction code would make a reservation while building the window, before a user activates the button.

The lambda refers to the existing model and controls. Calling methods on those objects can change their state without assigning different objects to the local variable names. Keeping the references stable lets later events keep using the same model and view for this window session.


### Keep the count in the model and its presentation in the view

The **model** is the ordinary Java object that stores the application state and applies its rules. The **view** is the interface that presents that state. Our `SeatCounter` is the model; the label and button belong to the view.

Here is the complete model class from the example:

```java
class SeatCounter {
    private int remaining;
    public SeatCounter(int remaining) { this.remaining = remaining; }
    public int getRemaining() { return remaining; }
    public void reserve() {
        if (remaining > 0) { remaining--; }
    }
}
```

The constructor receives the starting count and stores it in the private `remaining` field. For this small class, the caller must supply a nonnegative starting count. The constructor does not reject a negative argument. The canonical example satisfies that requirement by starting at three.

`getRemaining` returns the stored count. `reserve` changes it only when it is greater than zero. The decrement `remaining--` subtracts one. At zero, the guarded statement is skipped, so another direct call cannot decrease the count below zero from a valid starting state.

The model contains no Label or Button. Its rule can be checked in an ordinary Java cell without opening a window. The view reads the model when creating its initial message:

```java
    Label status = new Label("Remaining: " + model.getRemaining());
```

The label's sentence is a presentation of the count, not the authoritative count itself. The program should ask the model how many seats remain rather than parse a number back out of the label. Keeping these roles separate lets the same model rule remain valid when the interface's wording or layout changes.


### Change the model before refreshing the display

An **event-driven state update** changes application state in response to an event and then refreshes the visible result. Our handler performs these statements in order:

```java
        model.reserve();
        status.setText("Remaining: " + model.getRemaining());
        reserve.setDisable(model.getRemaining() == 0);
```

First, the model applies its reservation rule. Second, the label receives text built from the updated count. Third, the button's availability is adjusted for the new state. Each step uses the result of the preceding step; the display should describe the state after the requested operation.

Starting from three seats, the first valid activation changes the model to two and refreshes the label to Remaining: 2. Later activations can reach zero. Updating only the model would leave old text on screen. Updating only the label would leave the model holding an old count, so later decisions would use the wrong state.

JavaFX calls these action handlers on the JavaFX Application Thread introduced earlier in this module. That is the appropriate place for these short control updates. A long search, file operation, or waiting loop inside the handler could delay other interface events; this lesson keeps each response short.

The initial `Counter ready: 3` line in the notebook reports construction of the example. It does not print again after every activation. The native window supplies the visible interaction results, while separate direct model checks can inspect the stored count. Those are different forms of evidence.


### Make unavailable actions visible and keep the model guard

**Control enablement** determines whether a control is available for ordinary user interaction. In the reserve handler, this statement updates that availability:

```java
        reserve.setDisable(model.getRemaining() == 0);
```

The comparison produces a boolean. At zero, it is `true`, so `setDisable` disables the button. With seats remaining, it is `false`, so the button is not disabled by this setting. The method's name describes disabling: passing true prevents the action rather than enabling it.

In the three-seat example, the activation that reaches zero also displays Remaining: 0 and disables Reserve one. A later attempt to activate that disabled control should leave the display unchanged. The number in the label explains the boundary without relying only on the button's appearance or color.

The disabled button and the model's guard serve different purposes. Disabling guides the user away from an unavailable action. The guard still protects the rule if ordinary Java code calls `reserve` directly. A disabled control is not a substitute for a valid model operation.

The initial interface must also agree with its model before the first action. The canonical example starts with three seats, so an enabled button is appropriate. A variation with a different initial state must establish the appropriate initial availability as well as update it after events. It cannot rely on a first reservation event to make an already empty model visible correctly.


### Separate registration from activation

<details class="animation-panel">
<summary>Show or hide the animation</summary>
<p><img src="media/03_events_and_model_state/register_then_activate.gif" alt="Registration leaves the model unchanged; a later button activation calls the handler, updates the model, and refreshes the controls." width="960" style="max-width:100%;height:auto;"></p>
</details>
<p>This loop lasts about 10 seconds.</p>
<p>Creating the handler supplies behavior for a later event. The model remains unchanged during registration. When an enabled button is activated, JavaFX calls the handler, which applies the model rule and refreshes the controls.</p>
<p><a href="media/03_events_and_model_state/register_then_activate_still.png">View the final state as a still image</a>.</p>


### Follow the model and view to the boundary

<details class="animation-panel">
<summary>Show or hide the animation</summary>
<p><img src="media/03_events_and_model_state/model_view_boundary.gif" alt="The count reaches zero; the Label shows zero and the button becomes disabled while the model guard prevents a negative count." width="960" style="max-width:100%;height:auto;"></p>
</details>
<p>This loop lasts about 12.5 seconds.</p>
<p>The model changes first. The Label then reads the new count, and the button’s disabled setting follows that same state. At zero, the model guard and the unavailable action serve separate purposes.</p>
<p><a href="media/03_events_and_model_state/model_view_boundary_still.png">View the final state as a still image</a>.</p>


### Check the keyboard destination before activating

**Keyboard focus** identifies the control that currently receives keyboard input. **Keyboard activation** uses a key to request that control's action. Focus and activation are separate: moving focus to a button does not reserve a seat.

In the Workspace window, Tab moves focus among available controls, and Shift+Tab moves through that order in reverse. The focus indicator identifies the current keyboard destination. When the enabled Reserve one button has focus, Space can activate it and reach the same action handler as a pointer click.

Focus is also different from enablement. An enabled button can be available even when another control has focus. A disabled action must not become available merely because a user tries another input method. The model rule, label refresh, and disabled boundary should remain consistent for both pointer and keyboard use.

Testing should therefore include reaching the button with the keyboard, activating it, observing the updated text, and trying the action after the boundary disables it. A pointer-only test would not show whether the keyboard path is usable. A correctly colored button would not show whether the model kept its valid state.

We now have the complete interaction: the user activates a control, JavaFX calls its registered handler, the model applies its rule, and the view reports the new state and available actions. The worked program assembles those pieces in one native window.


### Prepare the supplied notebook support

This is supplied course support for running JavaFX inside IJava. Run it once after starting or restarting this notebook's Java kernel. The message `FX ready` means the support has initialized JavaFX and completed an operation on its application thread. Open the Workspace **Desktop** view to see the windows created by later cells.

`Fx.run(() -> { ... })` performs the enclosed UI work on the JavaFX Application Thread and waits for that short operation to finish. Use it for reading as well as changing a live window or its controls. `Fx.closeWindows()` hides the windows created by this kernel before another example opens its own. `Fx.start()` is safe to call again; it keeps JavaFX available after the last window closes.

The implementation below is provided runtime support. You do not need to write its thread-coordination machinery for this lesson. A thread is one sequence of execution; Module 13 studies how to coordinate more than one. Here your responsibility is to use the documented support operations and keep UI work short. The support uses a completion signal, a time limit, and an error holder so a later cell does not silently continue after unfinished or failed UI work.

Do not call `Platform.exit()` during notebook practice. That ends the toolkit for this kernel; restart the kernel and rerun setup if you do so. Closing a window is different from ending the toolkit. If setup reports a display error, check that the Workspace Desktop is running, then restart the kernel and rerun setup. A JavaFX window appears in the Desktop, not as an inline notebook control.


In [ ]:
import javafx.application.Platform;
import javafx.stage.Window;
import java.util.ArrayList;
import java.util.concurrent.CountDownLatch;
import java.util.concurrent.TimeUnit;
import java.util.concurrent.atomic.AtomicReference;
class Fx {
    static void run(Runnable action) throws InterruptedException {
        if (Platform.isFxApplicationThread()) {
            action.run();
            return;
        }
        CountDownLatch done = new CountDownLatch(1);
        AtomicReference<Throwable> failure = new AtomicReference<Throwable>();
        Platform.runLater(() -> {
            try { action.run(); }
            catch (Throwable error) { failure.set(error); }
            finally { done.countDown(); }
        });
        if (!done.await(10, TimeUnit.SECONDS)) {
            throw new IllegalStateException("FX operation timed out; restart the kernel.");
        }
        if (failure.get() != null) { throw new RuntimeException(failure.get()); }
    }
    static void start() throws InterruptedException {
        try { Platform.startup(() -> Platform.setImplicitExit(false)); }
        catch (IllegalStateException alreadyStarted) {
            // This call is also safe when this kernel already started JavaFX.
        }
        run(() -> Platform.setImplicitExit(false));
    }
    static void closeWindows() throws InterruptedException {
        run(() -> {
            for (Window window : new ArrayList<Window>(Window.getWindows())) {
                window.hide();
            }
        });
    }
}
Fx.start();
System.out.println("FX ready");


### Read a complete counter without a boundary

A staff member can use a small counter to record each request for help. This first complete program starts at zero and adds one for each activation. It introduces the connection between an ordinary model and its displayed count before the bounded reservation example.

`ClickCount` stores the number. Its `addOne` method changes that state, while `getCount` lets the view read it. After creating the model, Label, and Button, the program registers a handler. The handler changes the model first and then builds the Label's new text from the updated count. The VBox arranges the two controls using the layout settings from the preceding lesson.

This complete reading example uses the supplied support above. You may copy it into a Java work cell to inspect the window.

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
class ClickCount {
    private int count;
    public ClickCount() { count = 0; }
    public void addOne() { count++; }
    public int getCount() { return count; }
}
Fx.closeWindows();
Fx.run(() -> {
    ClickCount model = new ClickCount();
    Label status = new Label("Clicks: " + model.getCount());
    Button add = new Button("Add one");
    add.setOnAction(event -> {
        model.addOne();
        status.setText("Clicks: " + model.getCount());
    });
    VBox root = new VBox(12, status, add);
    root.setPadding(new Insets(20));
    Stage stage = new Stage();
    stage.setTitle("Click Counter");
    stage.setScene(new Scene(root, 340, 220));
    stage.show();
    System.out.println("Initial count: " + model.getCount());
});
```

Expected initial output:

```text
Initial count: 0
```

The native window is titled Click Counter and initially shows Clicks: 0 above Add one. Each activation updates the visible count. The initial console report does not print again because it is outside the handler. Close this window before running another complete example. This model has no upper-limit rule; the next program adds a guard for a limited resource.


## Worked Example

### Reserve seats without passing zero

A campus welcome session has three available seats. The window must show the remaining count, accept one reservation per activation, and make Reserve one unavailable when the count reaches zero. This is one local model and window; it does not coordinate a shared booking service.

**Create the state and its initial view.** The imports make the required JavaFX types available by their short names. `SeatCounter` stores a valid starting count and guards its decrement. After closing any previous window, the supplied operation creates a fresh model with three seats, a Label that reads its count, and the Button.

**Register the later update.** Connect the button's action to the three already explained steps: apply the model rule, refresh the Label from the resulting count, and update the disabled setting. The construction code registers this behavior; it does not reserve a seat itself.

**Assemble and inspect.** Put the Label and Button in a VBox with spacing 12 and padding 20. Attach that layout to a Scene with a 380 by 220 content area, give the Stage its title, and show it. The print at the end reports the initial model count. Later actions are visible in the window.


In [ ]:
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
class SeatCounter {
    private int remaining;
    public SeatCounter(int remaining) { this.remaining = remaining; }
    public int getRemaining() { return remaining; }
    public void reserve() {
        if (remaining > 0) { remaining--; }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    SeatCounter model = new SeatCounter(3);
    Label status = new Label("Remaining: " + model.getRemaining());
    Button reserve = new Button("Reserve one");
    reserve.setOnAction(event -> {
        model.reserve();
        status.setText("Remaining: " + model.getRemaining());
        reserve.setDisable(model.getRemaining() == 0);
    });
    VBox root = new VBox(12, status, reserve);
    root.setPadding(new Insets(20));
    stage.setTitle("Seat Counter");
    stage.setScene(new Scene(root, 380, 220));
    stage.show();
    System.out.println("Counter ready: " + model.getRemaining());
});


Expected output:

```text
Counter ready: 3
```

Construction reports three seats and shows Remaining: 3. The first three activations display 2, 1, and 0. The third also disables Reserve one. A later attempt through that disabled control leaves the count at zero. The model’s own guard also prevents a direct reserve call from making the count negative.


Open the Workspace Desktop to inspect Seat Counter. Activate Reserve one and compare the new text with the preceding count. Continue to the zero boundary and attempt another activation. The label must continue to describe the model, and the disabled button must show that another reservation is unavailable. Close the native window when finished. A fresh complete run constructs a new model and window; it does not restore state by editing the old Label.

For the keyboard tests below, make the native window active. Tab and Shift+Tab move keyboard focus between eligible controls. Look for the visible focus indicator, then use Space on the enabled button. A pointer click on the enabled button also normally leaves it focused for the following Space action. Keep keyboard focus, button availability, and the model's count separate in your observations.


## Guided Practice

The guided program models a small session that starts with two seats. Work through prediction, completion, initial-state changes, and repair. Keep each prediction when you record the actual result. Run the supplied support first, and use complete programs in the Java work cells. Empty work cells are places to write a solution, not completed examples.

For each sequence, inspect the initial console report and the actual Desktop. Record the label and button state after each input. Close the native window before creating a fresh version. An unchanged console line does not mean that later actions left the interface unchanged.


### Predict a two-seat reservation sequence

Before running the complete program below, predict its initial console line and visible Label. Decide whether handler registration itself uses a seat. Then predict the Label and button state after one pointer click, Space on the focused Reserve one button, and one more click attempt. Record the predictions first, then run the complete cell.


In [ ]:
Initial output and registration:
Your response

Pointer, Space, extra-click predictions:
Your response


In [ ]:
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
class SeatCounter {
    private int remaining;
    public SeatCounter(int remaining) { this.remaining = remaining; }
    public int getRemaining() { return remaining; }
    public void reserve() {
        if (remaining > 0) { remaining--; }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    SeatCounter model = new SeatCounter(2);
    Label status = new Label("Remaining: " + model.getRemaining());
    Button reserve = new Button("Reserve one");
    reserve.setOnAction(event -> {
        model.reserve();
        status.setText("Remaining: " + model.getRemaining());
        reserve.setDisable(model.getRemaining() == 0);
    });
    VBox root = new VBox(12, status, reserve);
    root.setPadding(new Insets(20));
    stage.setTitle("Reservation Desk");
    stage.setScene(new Scene(root, 380, 220));
    stage.show();
    System.out.println("Counter ready: " + model.getRemaining());
});


Record the actual initial console line and Label. Perform the pointer click, focused Space, and extra click attempt in that order. Record the Label, button state, and any new stdout after each input. Close the native window, keep your original predictions, and explain any correction.


In [ ]:
Initial actual output:
Your response

Actual sequence and enablement:
Your response

New stdout, native close, corrections:
Your response


Name the object that owns the authoritative count. Trace the model change, Label refresh, and button-state update after one action. Explain why changing the model does not refresh a Label automatically, why the captured references can remain unchanged, and why this short live-interface work belongs on the JavaFX Application Thread.


In [ ]:
Authoritative state and update order:
Your response

Stable references and application-thread work:
Your response


<details>
<summary>Show answer</summary>

The initial console prints `Counter ready: 2`. Reservation Desk starts with `Remaining: 2` and an enabled Reserve one button. Registering the lambda with `setOnAction` stores behavior for a future action; it does not call `model.reserve()` during construction.

A pointer click changes the model to 1, refreshes the label to `Remaining: 1`, and leaves the button enabled. With Reserve one focused, Space requests the same action: the model reaches 0, the label becomes `Remaining: 0`, and the button becomes disabled. Another native click on that disabled control requests no handler action, so the label stays at zero. These later UI actions add no console output.

The model values follow from the exact method and callback statements; the label and button state are the visible observations. The class also checks `remaining > 0` before decrementing. The model guard and the disabled control serve different purposes: one protects the stored value when the method is called, and the other controls whether the user can request the action through this button.

The `SeatCounter` object owns the private remaining value. The Label displays text copied from the model; it does not automatically track later field changes. The handler first calls the model method, then refreshes the Label, then sets the button state from the new remaining value.

Its lambda captures the model and control references. Those local variables keep referring to the same objects while the objects change state.

The supplied `Fx.run` performs construction and initial UI reads on the JavaFX Application Thread; the registered action callback also performs its UI updates there. Both pointer activation and Space on the focused button reach that registered behavior.

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
class SeatCounter {
    private int remaining;
    public SeatCounter(int remaining) { this.remaining = remaining; }
    public int getRemaining() { return remaining; }
    public void reserve() {
        if (remaining > 0) { remaining--; }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    SeatCounter model = new SeatCounter(2);
    Label status = new Label("Remaining: " + model.getRemaining());
    Button reserve = new Button("Reserve one");
    reserve.setOnAction(event -> {
        model.reserve();
        status.setText("Remaining: " + model.getRemaining());
        reserve.setDisable(model.getRemaining() == 0);
    });
    VBox root = new VBox(12, status, reserve);
    root.setPadding(new Insets(20));
    stage.setTitle("Reservation Desk");
    stage.setScene(new Scene(root, 380, 220));
    stage.show();
    System.out.println("Counter ready: " + model.getRemaining());
});
```

Expected output:

```text
Counter ready: 2
```

Common error: Treating handler registration as an immediate method call. Assuming later button actions print another initial report. Predicting button state without tracing the model update.

</details>


### Complete the known handler

Reconstruct the two-seat program by replacing the four uppercase markers with `setOnAction`, `reserve`, `setText`, and `setDisable`, each in its appropriate place. Explain your choices and reconstruct the expected action sequence before running. The displayed source is incomplete and must be edited.

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
class SeatCounter {
    private int remaining;
    public SeatCounter(int remaining) { this.remaining = remaining; }
    public int getRemaining() { return remaining; }
    public void reserve() {
        if (remaining > 0) { remaining--; }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    SeatCounter model = new SeatCounter(2);
    Label status = new Label("Remaining: " + model.getRemaining());
    Button reserve = new Button("Reserve one");
    reserve.HANDLER_METHOD(event -> {
        model.MODEL_UPDATE();
        status.LABEL_UPDATE("Remaining: " + model.getRemaining());
        reserve.BUTTON_STATE(model.getRemaining() == 0);
    });
    VBox root = new VBox(12, status, reserve);
    root.setPadding(new Insets(20));
    stage.setTitle("Reservation Desk");
    stage.setScene(new Scene(root, 380, 220));
    stage.show();
    System.out.println("Counter ready: " + model.getRemaining());
});
```

After recording the choices, copy the completed program into the Java work cell and run it.


In [ ]:
Four replacements and reasons:
Your response

Reconstructed action sequence:
Your response


Record the completed program’s initial output and actual pointer, focused-Space, and disabled-click results. Close the native window. Explain why the model update must precede the view updates.


In [ ]:
Actual output and sequence:
Your response

Native close and update-order explanation:
Your response


<details>
<summary>Show answer</summary>

Use `setOnAction` to register the callback, `reserve` to update the model, `setText` to refresh the Label and `setDisable` to set the control state.

The initial console prints `Counter ready: 2`. Reservation Desk starts with `Remaining: 2` and an enabled Reserve one button. Registering the lambda with `setOnAction` stores behavior for a future action; it does not call `model.reserve()` during construction.

A pointer click changes the model to 1, refreshes the label to `Remaining: 1`, and leaves the button enabled. With Reserve one focused, Space requests the same action: the model reaches 0, the label becomes `Remaining: 0`, and the button becomes disabled. Another native click on that disabled control requests no handler action, so the label stays at zero. These later UI actions add no console output.

The model values follow from the exact method and callback statements; the label and button state are the visible observations. The class also checks `remaining > 0` before decrementing. The model guard and the disabled control serve different purposes: one protects the stored value when the method is called, and the other controls whether the user can request the action through this button.

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
class SeatCounter {
    private int remaining;
    public SeatCounter(int remaining) { this.remaining = remaining; }
    public int getRemaining() { return remaining; }
    public void reserve() {
        if (remaining > 0) { remaining--; }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    SeatCounter model = new SeatCounter(2);
    Label status = new Label("Remaining: " + model.getRemaining());
    Button reserve = new Button("Reserve one");
    reserve.setOnAction(event -> {
        model.reserve();
        status.setText("Remaining: " + model.getRemaining());
        reserve.setDisable(model.getRemaining() == 0);
    });
    VBox root = new VBox(12, status, reserve);
    root.setPadding(new Insets(20));
    stage.setTitle("Reservation Desk");
    stage.setScene(new Scene(root, 380, 220));
    stage.show();
    System.out.println("Counter ready: " + model.getRemaining());
});
```

Expected output:

```text
Counter ready: 2
```

Common error: Putting a control method on the model object. Reading the old model value before the reservation. Leaving placeholder names in runnable code.

</details>


### Handle a zero-seat starting state

The complete starter below begins with two seats. Change the starting count to zero. Add an initial `setDisable` call after creating the Button, using the same model-based condition as the handler. Preserve the handler’s later check. Predict the initial Label, console line, button state, and result of a pointer click attempt. After recording the prediction, make these edits and run the complete program.


In [ ]:
Edits and predicted zero-start result:
Your response


In [ ]:
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
class SeatCounter {
    private int remaining;
    public SeatCounter(int remaining) { this.remaining = remaining; }
    public int getRemaining() { return remaining; }
    public void reserve() {
        if (remaining > 0) { remaining--; }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    SeatCounter model = new SeatCounter(2);
    Label status = new Label("Remaining: " + model.getRemaining());
    Button reserve = new Button("Reserve one");
    reserve.setOnAction(event -> {
        model.reserve();
        status.setText("Remaining: " + model.getRemaining());
        reserve.setDisable(model.getRemaining() == 0);
    });
    VBox root = new VBox(12, status, reserve);
    root.setPadding(new Insets(20));
    stage.setTitle("Reservation Desk");
    stage.setScene(new Scene(root, 380, 220));
    stage.show();
    System.out.println("Counter ready: " + model.getRemaining());
});


Record the actual zero-start output, Label, initial disabled state, and click-attempt result. Close the native window. Explain why a check only inside the handler misses the initial zero case.


In [ ]:
Actual zero-start output and click result:
Your response

Native close and initial-check explanation:
Your response


### Test one starting seat

Preserve the zero-start observations. Change only the model’s starting count to one, keeping both availability checks. Predict the initial result, focused Space activation, and an extra click attempt after the button disables. After recording the prediction, update the Java work cell above and run the complete program as a fresh window.


In [ ]:
Predicted one-start output and sequence:
Your response


Record the one-start initial output and visible state, the focused-Space result, and the disabled-click result. Close the window. Compare the roles of the initial check and the handler’s later check.


In [ ]:
Actual one-start output and sequence:
Your response

Native close and two checks:
Your response


<details>
<summary>Show answer</summary>

Add `reserve.setDisable(model.getRemaining() == 0);` immediately after constructing Reserve one. With zero seats, initial stdout is `Counter ready: 0`; the Label reads `Remaining: 0`, and the button is already disabled. A click attempt changes nothing. The handler has not run yet, so its later check alone cannot set this initial state. The one-seat comparison starts enabled, reaches zero through focused Space, refreshes the Label and disables the button. Both programs close normally. The model and handler definitions are otherwise unchanged.

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
class SeatCounter {
    private int remaining;
    public SeatCounter(int remaining) { this.remaining = remaining; }
    public int getRemaining() { return remaining; }
    public void reserve() {
        if (remaining > 0) { remaining--; }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    SeatCounter model = new SeatCounter(0);
    Label status = new Label("Remaining: " + model.getRemaining());
    Button reserve = new Button("Reserve one");
    reserve.setDisable(model.getRemaining() == 0);
    reserve.setOnAction(event -> {
        model.reserve();
        status.setText("Remaining: " + model.getRemaining());
        reserve.setDisable(model.getRemaining() == 0);
    });
    VBox root = new VBox(12, status, reserve);
    root.setPadding(new Insets(20));
    stage.setTitle("Reservation Desk");
    stage.setScene(new Scene(root, 380, 220));
    stage.show();
    System.out.println("Counter ready: " + model.getRemaining());
});
```

Expected output:

```text
Counter ready: 0
```

Common error: Adding the initial check before constructing the Button. Keeping the button enabled at zero until its first event. Removing the handler check after adding the initial check.

**Additional test: `modify_initial_one`.** A fresh one-seat program prints `Counter ready: 1`, begins with `Remaining: 1` and an enabled button, then focused Space reaches `Remaining: 0` and disables it. The extra native click leaves zero unchanged. This checks both the initial nonzero state and the later boundary transition.

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
class SeatCounter {
    private int remaining;
    public SeatCounter(int remaining) { this.remaining = remaining; }
    public int getRemaining() { return remaining; }
    public void reserve() {
        if (remaining > 0) { remaining--; }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    SeatCounter model = new SeatCounter(1);
    Label status = new Label("Remaining: " + model.getRemaining());
    Button reserve = new Button("Reserve one");
    reserve.setDisable(model.getRemaining() == 0);
    reserve.setOnAction(event -> {
        model.reserve();
        status.setText("Remaining: " + model.getRemaining());
        reserve.setDisable(model.getRemaining() == 0);
    });
    VBox root = new VBox(12, status, reserve);
    root.setPadding(new Insets(20));
    stage.setTitle("Reservation Desk");
    stage.setScene(new Scene(root, 380, 220));
    stage.show();
    System.out.println("Counter ready: " + model.getRemaining());
});
```

Expected output:

```text
Counter ready: 1
```

</details>


### Repair a stale Label

The displayed diagnostic deliberately omits the Label refresh while retaining the model update and button check. Predict what the faulty Label would show after one pointer click and then focused Space. Explain why its text could disagree with the disabled button. Identify the missing statement and its place in the handler before editing.

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
class SeatCounter {
    private int remaining;
    public SeatCounter(int remaining) { this.remaining = remaining; }
    public int getRemaining() { return remaining; }
    public void reserve() {
        if (remaining > 0) { remaining--; }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    SeatCounter model = new SeatCounter(2);
    Label status = new Label("Remaining: " + model.getRemaining());
    Button reserve = new Button("Reserve one");
    reserve.setOnAction(event -> {
        model.reserve();
        reserve.setDisable(model.getRemaining() == 0);
    });
    VBox root = new VBox(12, status, reserve);
    root.setPadding(new Insets(20));
    stage.setTitle("Reservation Desk");
    stage.setScene(new Scene(root, 380, 220));
    stage.show();
    System.out.println("Counter ready: " + model.getRemaining());
});
```

After recording the diagnosis, copy the program into the Java work cell, restore the Label refresh after the model update, and run the complete repaired program.


In [ ]:
Predicted stale Label and disagreement:
Your response

Repair statement and placement:
Your response


Record the repaired program’s initial output and actual pointer, focused-Space, and disabled-click sequence. Close the window. Explain how the repair keeps the displayed count consistent with the model.


In [ ]:
Actual repaired output and sequence:
Your response

Native close and explanation:
Your response


<details>
<summary>Show answer</summary>

The faulty draft initially prints `Counter ready: 2` and shows `Remaining: 2`. After the first click, the label still reads `Remaining: 2`; after the focused Space activation, the button disables but the label still reads `Remaining: 2`. The model method and disable condition remain present, while the Label never receives new text. Restore `status.setText("Remaining: " + model.getRemaining());` after `model.reserve()` and before `reserve.setDisable(...)`. The complete repair is the prediction program: the Label progresses to `Remaining: 1` and then `Remaining: 0`, and the extra click on the disabled control changes nothing. The visible stale text is the concrete fault; normal completion of the notebook cell does not establish that later events update the display correctly.

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
class SeatCounter {
    private int remaining;
    public SeatCounter(int remaining) { this.remaining = remaining; }
    public int getRemaining() { return remaining; }
    public void reserve() {
        if (remaining > 0) { remaining--; }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    SeatCounter model = new SeatCounter(2);
    Label status = new Label("Remaining: " + model.getRemaining());
    Button reserve = new Button("Reserve one");
    reserve.setOnAction(event -> {
        model.reserve();
        status.setText("Remaining: " + model.getRemaining());
        reserve.setDisable(model.getRemaining() == 0);
    });
    VBox root = new VBox(12, status, reserve);
    root.setPadding(new Insets(20));
    stage.setTitle("Reservation Desk");
    stage.setScene(new Scene(root, 380, 220));
    stage.show();
    System.out.println("Counter ready: " + model.getRemaining());
});
```

Expected output:

```text
Counter ready: 2
```

Common error: Changing the initial Label text instead of refreshing it in the callback. Refreshing the Label before calling the model method. Assuming a normally completed cell proves all future button behavior.

</details>


## Independent Practice

Build a points counter for a campus activity. Separate its point rule from the visible controls, then test how Reset restores a usable interface from different states. Plan the model and both handlers before writing the complete program.


### Build a points counter with reset

Create `PointCounter` with private points initially 0, `getPoints`, `addTwo` that stops at 4, and `reset` to 0. Build a Point Counter window with a 380 by 260 Scene, a VBox gap of 12 and padding of 20. Display `Points: n` followed by Add two and Reset buttons. Each handler must change the model and then refresh the Label. Disable Add two at 4; Reset must restore 0 and re-enable Add two. Include all required imports and the complete model class, and use the supplied `Fx.closeWindows` and `Fx.run` operations. Show the window and print only the initial report `Counter ready: 0`. Test two pointer activations, another click attempt while Add two is disabled, Reset, and Space on the keyboard-focused Add two button. After Reset, Shift+Tab moves focus from Reset back to Add two. Close the window. Explain the model change separately from the Label refresh.


Plan the model and both handlers, and predict every state in the required full sequence. After recording the plan, write and run your complete program in the Java work cell.


In [ ]:
Model and handler plan:
Your response

Predicted full action sequence:
Your response


Record the actual initial console line and each Label and Add two state in the required sequence: two pointer activations, an extra disabled-click attempt, Reset, then Shift+Tab and Space on Add two. Close the window. Separate model changes from Label refreshes, and distinguish initial-only stdout from later UI changes.


In [ ]:
Actual output and full UI sequence:
Your response

Model versus view, stdout, native close:
Your response


### Test Reset at zero

Preserve the first sequence and its observations. Predict the result of Reset immediately in a fresh zero-point window, followed by Shift+Tab and Space on Add two. Record the Label and Add two state you expect after each action. Then rerun the complete program in the work cell above and perform this sequence.


In [ ]:
Reset-at-zero and keyboard prediction:
Your response


Record the actual initial report, reset-at-zero result, keyboard result, and enablement after each action. Close the native window. Explain what this test adds to Reset at the upper limit.


In [ ]:
Actual reset-at-zero and keyboard result:
Your response

Native close and comparison:
Your response


### Test Reset after one addition

Predict Add two, Reset, and Add two again in a fresh window. Include the Label and availability after each action. Keep earlier results, record this prediction, then rerun the complete program in the work cell above and perform the sequence.


In [ ]:
Add-reset-add prediction:
Your response


Record the actual add-reset-add sequence and native close. Compare these results with Reset at zero and at the upper limit. Explain why the fixed initial console line cannot establish every later visible state.


In [ ]:
Actual add-reset-add and close:
Your response

Boundary comparison and stdout limits:
Your response


<details>
<summary>Show answer</summary>

The complete program initially prints `Counter ready: 0` and shows `Points: 0`; both buttons are enabled.

The first two Add two clicks update the model and Label to 2 and 4. At 4, Add two is disabled, and another native click does not change the display.

Reset changes the model to 0, refreshes the Label to `Points: 0` and re-enables Add two. With Reset focused after its click, Shift+Tab selects Add two; Space reaches the same add handler and displays `Points: 2`.

No handler prints another initial report. The private model holds points, while each control update explicitly refreshes what the learner sees. Native close removes the window.

Testing Reset at zero checks that it leaves a valid initial state usable.

Testing Reset after one addition checks that reset works before the upper limit, too.

Recreating the whole program gives each test its own model, controls and window.

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
class PointCounter {
    private int points;
    public PointCounter() { points = 0; }
    public int getPoints() { return points; }
    public void addTwo() {
        if (points < 4) { points += 2; }
    }
    public void reset() { points = 0; }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    PointCounter model = new PointCounter();
    Label status = new Label("Points: " + model.getPoints());
    Button add = new Button("Add two");
    Button reset = new Button("Reset");
    add.setOnAction(event -> {
        model.addTwo();
        status.setText("Points: " + model.getPoints());
        add.setDisable(model.getPoints() == 4);
    });
    reset.setOnAction(event -> {
        model.reset();
        status.setText("Points: " + model.getPoints());
        add.setDisable(false);
    });
    VBox root = new VBox(12, status, add, reset);
    root.setPadding(new Insets(20));
    stage.setTitle("Point Counter");
    stage.setScene(new Scene(root, 380, 260));
    stage.show();
    System.out.println("Counter ready: " + model.getPoints());
});
```

Expected output:

```text
Counter ready: 0
```

Common error: Resetting the model without refreshing the Label. Resetting to zero but leaving Add two disabled. Updating the Label without updating the model.

</details>


<details>
<summary>Show answer</summary>
<details class="animation-panel">
<summary>Show or hide the animation</summary>
<p><img src="media/03_events_and_model_state/reset_keyboard_recovery.gif" alt="Reset restores zero points and enables Add two; Shift+Tab and Space then add two points before the window closes." width="960" style="max-width:100%;height:auto;"></p>
</details>
<p>This loop lasts about 12.5 seconds.</p>
<p>Reset changes the model to zero points, refreshes the Label, and makes Add two available again. Shift+Tab moves keyboard focus from Reset to Add two; Space can then activate the enabled action. Restoring availability and restoring model state are both necessary.</p>
<p><a href="media/03_events_and_model_state/reset_keyboard_recovery_still.png">View the final state as a still image</a>.</p>
</details>


## Summary

Creating a Button supplies a control; registering an action handler supplies behavior for a later activation. The handler can respond to pointer or keyboard input through the same model rule. Keep its work short so the interface can process other events.

The model owns the authoritative state. The Label presents it, and the button's disabled setting communicates whether an action is available. Change the model before refreshing those controls. Keep the model guard even when the interface disables an action.

Keyboard focus identifies the target of keyboard input. Enablement determines whether the action is available. Inspect both when testing a boundary and when restoring controls after Reset.


With answers closed, explain registration versus execution, model versus Label, and focus versus enablement. State the three reserve-handler operations in order and explain what each contributes.


In [ ]:
Registration/model/focus distinctions:
Your response

Three ordered handler operations:
Your response


<details>
<summary>Show answer</summary>

Registration stores the behavior for JavaFX to call after activation. The model owns the count; the Label contains a presentation of it. A handler must apply the model operation, refresh the Label from the updated getter, and adjust the button's disabled setting in that order.

Focus and enablement answer different questions: which control receives keyboard input, and whether its action is available. Reset must restore the model, refresh its displayed value, and make the other action available when the restored state allows it. A model guard still protects a direct method call without a window.

</details>


## Reflection

Transfer the relationship between a protected model, visible feedback, and later user actions to another campus task. Include a keyboard path and a way to check the rule without opening a window.


Choose a limited campus resource with an Add or Use action and a Reset or Return action. Describe the valid starting state, model guard, resulting text, and control-availability rules. Give one pointer test, one keyboard test, and one direct model test that does not need a window. Explain what evidence each test provides.


In [ ]:
Resource and state rules:
Your response

Pointer, keyboard, and direct-model tests:
Your response


## Supplemental Reading

- [JavaFX 21 ButtonBase API](https://openjfx.io/javadoc/21/javafx.controls/javafx/scene/control/ButtonBase.html) documents action-handler registration.
- [JavaFX 21 Button API](https://openjfx.io/javadoc/21/javafx.controls/javafx/scene/control/Button.html) describes button activation.
- [JavaFX 21 EventHandler API](https://openjfx.io/javadoc/21/javafx.base/javafx/event/EventHandler.html) defines the event callback contract.
- [JavaFX 21 Node API](https://openjfx.io/javadoc/21/javafx.graphics/javafx/scene/Node.html) documents disable and focus behavior.
